In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

/kaggle/input/datasets/mohdtahasayed2904/taha-12234/LLM_SFT_Train_17693.jsonl
/kaggle/input/datasets/mohdtahasayed2904/continue-2/adapter_config (3).json
/kaggle/input/datasets/mohdtahasayed2904/continue-2/tokenizer_config (4).json
/kaggle/input/datasets/mohdtahasayed2904/continue-2/optimizer (3).pt
/kaggle/input/datasets/mohdtahasayed2904/continue-2/trainer_state (3).json
/kaggle/input/datasets/mohdtahasayed2904/continue-2/chat_template.jinja
/kaggle/input/datasets/mohdtahasayed2904/continue-2/training_args (3).bin
/kaggle/input/datasets/mohdtahasayed2904/continue-2/adapter_model (3).safetensors
/kaggle/input/datasets/mohdtahasayed2904/continue-2/tokenizer (3).json
/kaggle/input/datasets/mohdtahasayed2904/continue-2/scheduler (3).pt
/kaggle/input/datasets/mohdtahasayed2904/continue-2/rng_state (3).pth


In [2]:
!pip install trl==0.12.0  # or whatever version your tutorial expects

!pip install -U bitsandbytes>=0.46.1

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 310.2/310.2 kB 6.5 MB/s eta 0:00:00:00:01


In [ ]:
# ============================================================
# RESUME QLoRA TRAINING FROM CHECKPOINT-400
# Model: Qwen2.5-Coder-1.5B-Instruct
# Training dataset: LLM_SFT_Train_17693.jsonl
# Hardware: Kaggle 2x T4, using GPU 0 as in the previous run
# ============================================================

import os
import json
import shutil
import torch
from pathlib import Path

os.environ["CUDA_VISIBLE_DEVICES"] = "0"
os.environ["TOKENIZERS_PARALLELISM"] = "false"

assert torch.cuda.is_available(), "CUDA GPU is unavailable."

print("=" * 70)
print("RESUMING QLoRA TRAINING FROM STEP 400")
print("=" * 70)
print("GPU:", torch.cuda.get_device_name(0))
print("PyTorch:", torch.__version__)

# ------------------------------------------------------------
# 1. IMPORTS
# ------------------------------------------------------------

from datasets import load_dataset
from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM,
    BitsAndBytesConfig,
    DataCollatorForLanguageModeling,
)
from trl import SFTConfig, SFTTrainer
from peft import LoraConfig

# ------------------------------------------------------------
# 2. CONFIGURATION
# ------------------------------------------------------------

MODEL_NAME = "Qwen/Qwen2.5-Coder-1.5B-Instruct"

EXPECTED_RESUME_STEP = 400
MAX_SEQ_LENGTH = 2048

OUTPUT_DIR = Path(
    "/kaggle/working/Qwen2.5-Coder-1.5B-QLoRA-RESUME-400"
)
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# ------------------------------------------------------------
# 3. FIND THE UPLOADED CHECKPOINT
# ------------------------------------------------------------

print("\nSearching for the uploaded checkpoint...")

SEARCH_ROOTS = [
    Path("/kaggle/input/datasets/mohdtahasayed2904/continue-2"),
    Path("/kaggle/input"),
    Path("/kaggle/working"),
]

checkpoint_source = None

for root in SEARCH_ROOTS:
    if not root.exists():
        continue

    for current_root, dirs, files in os.walk(root):
        dirs[:] = [
            d for d in dirs
            if d not in {".cache", "node_modules", ".git"}
        ]

        has_adapter = any(
            f.startswith("adapter_model")
            and f.endswith(".safetensors")
            for f in files
        )

        has_state = any(
            f.startswith("trainer_state")
            and f.endswith(".json")
            for f in files
        )

        has_optimizer = any(
            f.startswith("optimizer")
            and f.endswith(".pt")
            for f in files
        )

        if has_adapter and has_state and has_optimizer:
            checkpoint_source = Path(current_root)
            break

    if checkpoint_source is not None:
        break

if checkpoint_source is None:
    raise FileNotFoundError(
        "Could not locate the checkpoint. "
        "Ensure the dataset containing the Step-400 files is attached."
    )

print("Checkpoint source:", checkpoint_source)

# ------------------------------------------------------------
# 4. NORMALIZE CHECKPOINT FILENAMES
# ------------------------------------------------------------

CHECKPOINT_DIR = OUTPUT_DIR / "checkpoint-400"
CHECKPOINT_DIR.mkdir(parents=True, exist_ok=True)

FILE_MAPPING = {
    "adapter_config": "adapter_config.json",
    "adapter_model": "adapter_model.safetensors",
    "chat_template": "chat_template.jinja",
    "optimizer": "optimizer.pt",
    "rng_state": "rng_state.pth",
    "scheduler": "scheduler.pt",
    "tokenizer": "tokenizer.json",
    "tokenizer_config": "tokenizer_config.json",
    "trainer_state": "trainer_state.json",
    "training_args": "training_args.bin",
}

source_files = [
    p for p in checkpoint_source.iterdir()
    if p.is_file()
]

for prefix, destination_name in FILE_MAPPING.items():

    matches = [
        p for p in source_files
        if p.name.startswith(prefix)
    ]

    if not matches:
        if prefix in {
            "chat_template",
            "tokenizer",
            "tokenizer_config",
        }:
            print(f"Optional file not found: {prefix}")
            continue

        raise FileNotFoundError(
            f"Required checkpoint file missing: {prefix}"
        )

    source_file = max(matches, key=lambda p: len(p.name))
    destination_file = CHECKPOINT_DIR / destination_name

    shutil.copy2(source_file, destination_file)

    print(f"Copied: {source_file.name} -> {destination_name}")

# ------------------------------------------------------------
# 5. VERIFY STEP 400
# ------------------------------------------------------------

STATE_PATH = CHECKPOINT_DIR / "trainer_state.json"

with open(STATE_PATH, "r", encoding="utf-8") as f:
    saved_state = json.load(f)

saved_global_step = int(saved_state.get("global_step", -1))

print("\nSaved global step:", saved_global_step)

if saved_global_step != EXPECTED_RESUME_STEP:
    raise ValueError(
        f"Expected Step {EXPECTED_RESUME_STEP}, "
        f"but found Step {saved_global_step}. "
        "Training was not started."
    )

required_files = [
    "adapter_config.json",
    "adapter_model.safetensors",
    "optimizer.pt",
    "scheduler.pt",
    "rng_state.pth",
    "trainer_state.json",
    "training_args.bin",
]

for filename in required_files:
    path = CHECKPOINT_DIR / filename

    if not path.exists():
        raise FileNotFoundError(
            f"Missing required checkpoint file: {filename}"
        )

print("Checkpoint verification successful.")

# ------------------------------------------------------------
# 6. LOCATE THE ORIGINAL TRAINING DATASET
# ------------------------------------------------------------

DATASET_CANDIDATES = [
    Path(
        "/kaggle/input/datasets/mohdtahasayed2904/"
        "taha-12234/LLM_SFT_Train_17693.jsonl"
    ),
    Path(
        "/kaggle/input/datasets/mohdtahasayed2904/"
        "fine-tuning-start/LLM_SFT_Train_17693.jsonl"
    ),
]

DATASET_PATH = next(
    (p for p in DATASET_CANDIDATES if p.exists()),
    None,
)

if DATASET_PATH is None:
    for root in [
        Path("/kaggle/input"),
        Path("/kaggle/working"),
    ]:
        if root.exists():
            matches = list(
                root.rglob("LLM_SFT_Train_17693.jsonl")
            )
            if matches:
                DATASET_PATH = matches[0]
                break

if DATASET_PATH is None:
    raise FileNotFoundError(
        "LLM_SFT_Train_17693.jsonl was not found. "
        "Attach the original training dataset."
    )

print("\nTraining dataset:", DATASET_PATH)

# ------------------------------------------------------------
# 7. LOAD TOKENIZER
# ------------------------------------------------------------

print("\nLoading tokenizer...")

tokenizer = AutoTokenizer.from_pretrained(
    MODEL_NAME,
    trust_remote_code=True,
)

tokenizer.padding_side = "right"

if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

print("Tokenizer loaded.")
print("EOS:", tokenizer.eos_token)
print("PAD:", tokenizer.pad_token)

# ------------------------------------------------------------
# 8. LOAD TRAINING DATA
# ------------------------------------------------------------

print("\nLoading training records...")

dataset = load_dataset(
    "json",
    data_files=str(DATASET_PATH),
    split="train",
)

print("Training examples:", len(dataset))
print("Columns:", dataset.column_names)

if len(dataset) != 17693:
    raise ValueError(
        f"Expected 17693 training records; found {len(dataset)}."
    )

if "messages" not in dataset.column_names:
    raise ValueError(
        "The training dataset must contain the 'messages' column."
    )

# ------------------------------------------------------------
# 9. APPLY CHAT TEMPLATE
# ------------------------------------------------------------

def format_example(example):
    return {
        "text": tokenizer.apply_chat_template(
            example["messages"],
            tokenize=False,
            add_generation_prompt=False,
        )
    }

formatted_dataset = dataset.map(
    format_example,
    desc="Formatting conversations",
)

# ------------------------------------------------------------
# 10. TOKENIZE
# ------------------------------------------------------------

def tokenize_example(example):
    return tokenizer(
        example["text"],
        truncation=True,
        max_length=MAX_SEQ_LENGTH,
        padding="max_length",
        return_attention_mask=True,
    )

tokenized_dataset = formatted_dataset.map(
    tokenize_example,
    batched=True,
    remove_columns=formatted_dataset.column_names,
    desc="Tokenizing conversations",
)

print("Tokenization complete:", len(tokenized_dataset))

data_collator = DataCollatorForLanguageModeling(
    tokenizer=tokenizer,
    mlm=False,
)

# ------------------------------------------------------------
# 11. LOAD BASE MODEL IN 4-BIT
# ------------------------------------------------------------

print("\nLoading Qwen in 4-bit NF4...")

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_use_double_quant=True,
    bnb_4bit_compute_dtype=torch.bfloat16,
)

model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    quantization_config=bnb_config,
    device_map={"": 0},
    torch_dtype=torch.bfloat16,
    trust_remote_code=True,
)

model.config.use_cache = False

print("Base model loaded.")

# ------------------------------------------------------------
# 12. SAME LoRA CONFIGURATION
# ------------------------------------------------------------

lora_config = LoraConfig(
    r=16,
    lora_alpha=32,
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM",
    target_modules=[
        "q_proj",
        "k_proj",
        "v_proj",
        "o_proj",
        "gate_proj",
        "up_proj",
        "down_proj",
    ],
)

# ------------------------------------------------------------
# 13. SAME TRAINING CONFIGURATION
# ------------------------------------------------------------

sft_config = SFTConfig(
    output_dir=str(OUTPUT_DIR),

    num_train_epochs=1,

    per_device_train_batch_size=1,
    gradient_accumulation_steps=16,

    learning_rate=2e-4,
    weight_decay=0.01,

    optim="paged_adamw_8bit",

    warmup_steps=33,
    lr_scheduler_type="cosine",

    logging_steps=50,
    save_strategy="steps",
    save_steps=50,
    save_total_limit=10,

    bf16=True,
    fp16=False,

    gradient_checkpointing=True,
    gradient_checkpointing_kwargs={
        "use_reentrant": False
    },

    max_seq_length=MAX_SEQ_LENGTH,

    dataloader_num_workers=0,

    seed=42,
    data_seed=42,

    remove_unused_columns=False,
    report_to="none",
)

# ------------------------------------------------------------
# 14. INITIALIZE TRAINER
# ------------------------------------------------------------

print("\nInitializing trainer...")

trainer = SFTTrainer(
    model=model,
    args=sft_config,
    train_dataset=tokenized_dataset,
    processing_class=tokenizer,
    data_collator=data_collator,
    peft_config=lora_config,
)

# ------------------------------------------------------------
# 15. RESUME FROM CHECKPOINT 400
# ------------------------------------------------------------

print("\n" + "=" * 70)
print("RESUMING TRAINING FROM STEP 400")
print("=" * 70)
print("Checkpoint:", CHECKPOINT_DIR)
print("Saved step:", saved_global_step)
print("Original target: approximately 1106 steps")
print("=" * 70)

train_result = trainer.train(
    resume_from_checkpoint=str(CHECKPOINT_DIR)
)

# ------------------------------------------------------------
# 16. SAVE FINAL ADAPTER
# ------------------------------------------------------------

FINAL_ADAPTER_DIR = OUTPUT_DIR / "final_adapter"
FINAL_ADAPTER_DIR.mkdir(parents=True, exist_ok=True)

trainer.save_model(str(FINAL_ADAPTER_DIR))
tokenizer.save_pretrained(str(FINAL_ADAPTER_DIR))

# ------------------------------------------------------------
# 17. SAVE METRICS
# ------------------------------------------------------------

with open(
    OUTPUT_DIR / "training_metrics.json",
    "w",
    encoding="utf-8",
) as f:
    json.dump(
        train_result.metrics,
        f,
        indent=2,
        default=str,
    )

print("\n" + "=" * 70)
print("TRAINING FINISHED")
print("=" * 70)
print("Final global step:", trainer.state.global_step)
print("Final adapter:", FINAL_ADAPTER_DIR)

for key, value in train_result.metrics.items():
    print(f"{key}: {value}")

RESUMING QLoRA TRAINING FROM STEP 400
GPU: Tesla T4
PyTorch: 2.11.0+cu128

Searching for the uploaded checkpoint...
Checkpoint source: /kaggle/input/datasets/mohdtahasayed2904/continue-2
Copied: adapter_config (3).json -> adapter_config.json
Copied: adapter_model (3).safetensors -> adapter_model.safetensors
Copied: chat_template.jinja -> chat_template.jinja
Copied: optimizer (3).pt -> optimizer.pt
Copied: rng_state (3).pth -> rng_state.pth
Copied: scheduler (3).pt -> scheduler.pt
Copied: tokenizer_config (4).json -> tokenizer.json
Copied: tokenizer_config (4).json -> tokenizer_config.json
Copied: trainer_state (3).json -> trainer_state.json
Copied: training_args (3).bin -> training_args.bin

Saved global step: 400
Checkpoint verification successful.

Training dataset: /kaggle/input/datasets/mohdtahasayed2904/taha-12234/LLM_SFT_Train_17693.jsonl

Loading tokenizer...


config.json:   0%|          | 0.00/660 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/7.30k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

...

generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]

Base model loaded.

Initializing trainer...


[transformers] The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'bos_token_id': None, 'pad_token_id': 151643}.



RESUMING TRAINING FROM STEP 400
Checkpoint: /kaggle/working/Qwen2.5-Coder-1.5B-QLoRA-RESUME-400/checkpoint-400
Saved step: 400
Original target: approximately 1106 steps


Step,Training Loss
450,0.075860
500,0.071975
550,0.071968
600,0.070743


In [ ]:
import os

OUTPUT_DIR = "/kaggle/working/Qwen2.5-Coder-1.5B-QLoRA-SFT"

print("=" * 70)
print("CHECKING AVAILABLE CHECKPOINTS")
print("=" * 70)

if os.path.exists(OUTPUT_DIR):
    for item in sorted(os.listdir(OUTPUT_DIR)):
        path = os.path.join(OUTPUT_DIR, item)

        if os.path.isdir(path):
            print(f"\n📁 {item}/")

            for f in os.listdir(path):
                fp = os.path.join(path, f)
                size_mb = os.path.getsize(fp) / 1024**2

                print(f"   {f:<40} {size_mb:.2f} MB")
else:
    print("Output directory does not exist yet.")